<a href="https://colab.research.google.com/github/pachir1su/file_swordmaster/blob/main/ALL_swordmaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install gradio pypdf pydub moviepy reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s eta 0:00:00


In [11]:
import gradio as gr
from pypdf import PdfReader, PdfWriter
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os
import io

# 신규 라이브러리 (영상 추출 및 워터마크용)
from moviepy.editor import VideoFileClip
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

# ==========================================
# 1. PDF 분할 (PDF Swordmaster)
# ==========================================
def parse_page_string(page_string, total_pages):
    pages = set()
    if not page_string: return []
    for part in page_string.split(','):
        part = part.strip()
        if '-' in part:
            start, end = map(int, part.split('-'))
            pages.update(range(start, end + 1))
        else:
            pages.add(int(part))
    return sorted([p for p in pages if 1 <= p <= total_pages])

def process_pdf(pdf_file, page_string):
    if pdf_file is None: return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        total_pages = len(reader.pages)
        target_pages = parse_page_string(page_string, total_pages)
        if not target_pages: return None, f"유효하지 않은 입력입니다. (총 {total_pages}페이지)", gr.update(interactive=False)

        writer = PdfWriter()
        for p in target_pages: writer.add_page(reader.pages[p - 1])

        original_name = os.path.basename(pdf_file.name)
        name_only, ext = os.path.splitext(original_name)
        safe_page_str = page_string.replace(' ', '').replace(',', '_')
        output_path = f"{name_only}_pages_{safe_page_str}{ext}"

        with open(output_path, "wb") as f: writer.write(f)
        return output_path, f"성공적으로 추출되었습니다! (페이지: {target_pages})", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 2. 오디오 분할 (Audio Swordmaster)
# ==========================================
def parse_time_to_ms(time_str):
    if not time_str or str(time_str).strip() in ["", "0"]: return 0
    time_str = str(time_str).strip()
    try:
        if ':' in time_str:
            parts = time_str.split(':')
            if len(parts) == 2: return int((int(parts[0]) * 60 + float(parts[1])) * 1000)
            elif len(parts) == 3: return int((int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])) * 1000)
        else: return int(float(time_str) * 1000)
    except Exception: raise ValueError(f"잘못된 시간 형식: {time_str}")
    return 0

def process_audio(audio_file, start_time_str, end_time_str, remove_silence):
    if audio_file is None: return None, "오디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        audio = AudioSegment.from_file(audio_file)
        start_ms = parse_time_to_ms(start_time_str)
        end_ms = len(audio) if not end_time_str or str(end_time_str).strip() in ["", "0"] else parse_time_to_ms(end_time_str)

        if start_ms >= end_ms and end_ms != len(audio): return None, "시작이 종료보다 클 수 없습니다.", gr.update(interactive=False)
        audio = audio[start_ms:end_ms]

        if remove_silence:
            chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=-40)
            if chunks: audio = sum(chunks)

        original_name = os.path.basename(audio_file)
        name_only, ext = os.path.splitext(original_name)
        silence_tag = "_trimmed" if remove_silence else ""
        output_path = f"{name_only}_{start_ms//1000}s_to_{end_ms//1000}s{silence_tag}.mp3"

        audio.export(output_path, format="mp3")
        return output_path, "성공적으로 처리되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 3. PDF 병합 및 해제 (File Fusion)
# ==========================================
def process_fusion(pdf_files, password):
    if not pdf_files: return None, "합칠 PDF 파일들을 업로드해주세요.", gr.update(interactive=False)
    try:
        writer = PdfWriter()
        for pdf in pdf_files:
            reader = PdfReader(pdf.name)
            if reader.is_encrypted:
                if password:
                    reader.decrypt(password)
                else:
                    return None, f"잠긴 파일이 포함되어 있습니다. 비밀번호를 입력해주세요.", gr.update(interactive=False)

            for page in reader.pages:
                writer.add_page(page)

        original_name = os.path.basename(pdf_files[0].name)
        name_only, ext = os.path.splitext(original_name)
        others_count = len(pdf_files) - 1
        count_tag = f"_and_{others_count}others" if others_count > 0 else ""
        output_path = f"{name_only}{count_tag}_fusion.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)

        return output_path, f"총 {len(pdf_files)}개의 파일이 성공적으로 병합(해제)되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 4. 신규: 영상 음원 추출 (Audio Extractor)
# ==========================================
def process_video_extractor(video_file):
    if video_file is None: return None, "비디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        # 비디오 파일 불러오기
        clip = VideoFileClip(video_file.name)

        # 파일명 생성
        original_name = os.path.basename(video_file.name)
        name_only, ext = os.path.splitext(original_name)
        output_path = f"{name_only}_extracted_audio.mp3"

        # 오디오만 따로 저장 (logger=None으로 터미널 로그 숨김)
        clip.audio.write_audiofile(output_path, logger=None)
        clip.close()

        return output_path, "성공적으로 영상에서 소리만 추출되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 5. 신규: PDF 잠금 및 워터마크 (PDF Security)
# ==========================================
def process_pdf_security(pdf_file, new_password, watermark_text):
    if pdf_file is None: return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        writer = PdfWriter()

        # 텍스트 워터마크 생성 로직 (입력값이 있을 때만)
        watermark_reader = None
        if watermark_text and watermark_text.strip():
            packet = io.BytesIO()
            # A4 기준 중앙에 워터마크 그리기
            can = canvas.Canvas(packet, pagesize=A4)
            can.setFont("Helvetica", 60) # 영문 기본 폰트 사용
            can.setFillColorRGB(0.5, 0.5, 0.5, alpha=0.3) # 반투명한 회색

            # 페이지 중앙으로 이동 후 대각선으로 회전하여 텍스트 삽입
            can.translate(300, 400)
            can.rotate(45)
            can.drawCentredString(0, 0, watermark_text)
            can.save()

            packet.seek(0)
            watermark_reader = PdfReader(packet)
            watermark_page = watermark_reader.pages[0]

        # 모든 페이지를 돌며 합치기
        for page in reader.pages:
            if watermark_reader:
                page.merge_page(watermark_page) # 워터마크 도장 찍기
            writer.add_page(page)

        # 비밀번호를 입력했다면 암호 걸기
        if new_password and new_password.strip():
            writer.encrypt(new_password)

        # 파일명 생성
        original_name = os.path.basename(pdf_file.name)
        name_only, ext = os.path.splitext(original_name)
        output_path = f"{name_only}_secured.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)

        return output_path, "성공적으로 보안 처리가 완료되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 6. Web UI 구성
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as demo :
    gr.Markdown("# File Swordmaster")
    gr.Markdown("원하는 도구를 선택해 작업을 시작하세요!")

    with gr.Tab("PDF 분할"):
        gr.Markdown("### 원하는 페이지만 싹싹")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    pdf_input = gr.File(label="PDF 업로드", file_types=[".pdf"])
                    pdf_pages = gr.Textbox(label="추출 페이지 (예: 1, 3, 5-10)")
                    pdf_btn = gr.Button("자르기", variant="primary")
            with gr.Column(scale=1):
                with gr.Group():
                    pdf_out_file = gr.File(label="결과물")
                    pdf_out_msg = gr.Textbox(label="상태", interactive=False)
                    pdf_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False)
        pdf_btn.click(process_pdf, inputs=[pdf_input, pdf_pages], outputs=[pdf_out_file, pdf_out_msg, pdf_download_btn])

    with gr.Tab("오디오 분할"):
        gr.Markdown("### 정밀 타임라인 컷팅 및 무음 제거")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    audio_input = gr.Audio(label="오디오 업로드", type="filepath")
                    with gr.Row():
                        audio_start = gr.Textbox(label="시작 시간 (초/분:초)", value="0")
                        audio_end = gr.Textbox(label="종료 시간 (0이면 끝까지)", value="0")
                    audio_sil = gr.Checkbox(label="무음 구간 자동 제거", value=False)
                    audio_btn = gr.Button("자르기", variant="primary")
            with gr.Column(scale=1):
                with gr.Group():
                    audio_out_file = gr.File(label="결과물")
                    audio_out_msg = gr.Textbox(label="상태", interactive=False)
                    audio_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False)
        audio_btn.click(process_audio, inputs=[audio_input, audio_start, audio_end, audio_sil], outputs=[audio_out_file, audio_out_msg, audio_download_btn])

    with gr.Tab("PDF 병합 및 해제"):
        gr.Markdown("### 여러 PDF를 합치거나 암호 풀기")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    fusion_input = gr.File(label="여러 PDF 파일 업로드 (순서대로 병합됨)", file_types=[".pdf"], file_count="multiple")
                    fusion_pw = gr.Textbox(label="비밀번호 (잠긴 파일이 있을 경우 입력)", type="password")
                    fusion_btn = gr.Button("합치기 / 잠금해제", variant="primary")
            with gr.Column(scale=1):
                with gr.Group():
                    fusion_out_file = gr.File(label="결과물")
                    fusion_out_msg = gr.Textbox(label="상태", interactive=False)
                    fusion_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False)
        fusion_btn.click(process_fusion, inputs=[fusion_input, fusion_pw], outputs=[fusion_out_file, fusion_out_msg, fusion_download_btn])

    with gr.Tab("영상 음원 추출"):
        gr.Markdown("### 비디오 파일에서 소리만 추출")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    video_input = gr.File(label="영상 파일 업로드 (.mp4, .avi 등)")
                    video_btn = gr.Button("음원 추출하기", variant="primary")
            with gr.Column(scale=1):
                with gr.Group():
                    video_out_file = gr.File(label="결과물 (.mp3)")
                    video_out_msg = gr.Textbox(label="상태", interactive=False)
                    video_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False)
        video_btn.click(process_video_extractor, inputs=[video_input], outputs=[video_out_file, video_out_msg, video_download_btn])

    with gr.Tab("PDF 보안/워터마크"):
        gr.Markdown("### 문서 잠금 및 워터마크 싹싹")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    security_input = gr.File(label="원본 PDF 업로드", file_types=[".pdf"])
                    security_pw = gr.Textbox(label="새로 설정할 비밀번호 (공란이면 설정 안 함)", type="password")
                    security_wm = gr.Textbox(label="대각선 워터마크 텍스트 (예: CONFIDENTIAL)", placeholder="영문 사용 권장")
                    security_btn = gr.Button("보안 문서 만들기", variant="primary")
            with gr.Column(scale=1):
                with gr.Group():
                    security_out_file = gr.File(label="결과물")
                    security_out_msg = gr.Textbox(label="상태", interactive=False)
                    security_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False)
        security_btn.click(process_pdf_security, inputs=[security_input, security_pw, security_wm], outputs=[security_out_file, security_out_msg, security_download_btn])

demo.launch()

  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as demo :



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8456c1fe776fe61b15.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
